# 02. CNN 학습 (Colab GPU)

`signlight` 패키지(wheel)와 `index.csv`를 이용해 CNN을 학습하고, Kaggle val/test와 합성 변형셋으로 예측한다. 로컬 RF와 **같은 분할, 같은 변형 사진**으로 평가한다.

**실행 방법**
- Colab CLI: `scripts/colab_cnn.sh --notebook`이 wheel과 index.csv를 올리고 이 노트북을 실행한 뒤 결과를 받아 온다.
- Colab 웹: 런타임을 T4 GPU로 바꾸고, 왼쪽 파일 패널에서 `/content/`에 `signlight-0.1.0-py3-none-any.whl`(`uv build --wheel`로 생성)과 `index.csv`(`data/processed/index.csv`)를 올린 뒤 위에서부터 실행한다.

In [ ]:
# 설정: 학습할 변형과 시드. SMOKE=True면 클래스당 20장, 1 epoch로 동작만 확인한다.
VARIANTS = ['plain', 'aug']
SEEDS = [42, 43, 44]
EPOCHS = 15
SMOKE = False

In [ ]:
import glob, os, shutil, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor

ROOT = '/content/signlight'
os.environ['SIGNLIGHT_ROOT'] = ROOT
os.makedirs(f'{ROOT}/data/processed', exist_ok=True)
if os.path.exists('/content/index.csv'):
    shutil.copy('/content/index.csv', f'{ROOT}/data/processed/index.csv')
wheel = sorted(glob.glob('/content/signlight-*.whl') + glob.glob(f'{ROOT}/dist/signlight-*.whl'))[-1]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--ignore-requires-python', wheel], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow-heif'], check=True)
print('installed', os.path.basename(wheel))

In [ ]:
# 데이터: kagglehub(공개 데이터셋, 인증 불필요, Colab 캐시). /kaggle/input은 NFS라 대상 클래스만 로컬 디스크로 복사한다.
import kagglehub
from signlight import config as C

src = kagglehub.dataset_download('grassknoted/asl-alphabet')
t = time.time(); jobs = []
for cls in C.TARGET_CLASSES:
    s_dir = os.path.join(src, 'asl_alphabet_train', 'asl_alphabet_train', cls)
    d_dir = C.ASL_TRAIN_DIR / cls
    d_dir.mkdir(parents=True, exist_ok=True)
    jobs += [(os.path.join(s_dir, f), d_dir / f) for f in os.listdir(s_dir) if not (d_dir / f).exists()]
with ThreadPoolExecutor(32) as ex:
    list(ex.map(lambda j: shutil.copyfile(*j), jobs))
print('copied', len(jobs), 'files in', round(time.time() - t), 's')

In [ ]:
import tensorflow as tf
from signlight.data import load_index

idx = load_index()
missing = [p for p in idx['relpath'] if not (C.ASL_ROOT / p).exists()]
print('tf', tf.__version__, tf.config.list_physical_devices('GPU'))
print('index', len(idx), 'missing', len(missing))
assert not missing, missing[:5]

## 학습과 예측
각 (변형, 시드)마다 학습한 뒤 `kaggle_val`, `kaggle_test`, 합성 7조건 예측을 `results/preds/`에 저장한다.

In [ ]:
from signlight.cnn import predict, train

testsets = ['synth'] if SMOKE else ['kaggle_val', 'kaggle_test', 'synth']
for variant in VARIANTS:
    for seed in SEEDS:
        t = time.time()
        train(variant, seed, 1 if SMOKE else EPOCHS, per_class=20 if SMOKE else None)
        predict(variant, seed, testsets, per_class=5 if SMOKE else 100)
        print(f'== {variant} s{seed} done in {time.time() - t:.0f}s', flush=True)

## 결과 확인

In [ ]:
import json
import matplotlib.pyplot as plt
from signlight.evaluate import summarize
from signlight.report import seed_mean

s = summarize()
m = seed_mean(s[s['model'].str.startswith('cnn')])
print(m.pivot_table(index=['testset', 'condition'], columns='model', values='acc').round(3))

fig, ax = plt.subplots(figsize=(8, 4))
for f in sorted((C.RESULTS_DIR / 'logs').glob('cnn_*_history.json')):
    ax.plot(json.loads(f.read_text())['val_accuracy'], label=f.stem.replace('_history', ''))
ax.set_xlabel('epoch'); ax.set_ylabel('val accuracy (그룹 분할)'); ax.legend(fontsize=8)
plt.show()

In [ ]:
# 결과 묶기: Colab CLI는 이 파일을 받아 저장소에 푼다. Colab 웹에서는 파일 패널에서 내려받는다.
subprocess.run(f'cd {ROOT} && tar czf out.tgz results/preds results/logs $(ls models/cnn_*.keras)', shell=True, check=True)
print(os.path.getsize(f'{ROOT}/out.tgz') // 1024, 'KB ->', f'{ROOT}/out.tgz')